# 02 — PPO with Pixel Observations
Train a PPO agent on Super Mario Bros using pixel-based input.

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/contente/Documents/ENSTA/autonomous/CSC-52081-ContinousMountainCar


In [2]:
from src.wrappers import make_pixel_env
from src.agents import make_ppo_agent
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# Create environment
env = make_pixel_env()
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')

Observation space: (84, 84, 4)
Action space: 7


In [4]:
# Configure and create agent
config = PPOConfig(total_timesteps=2_000_000, policy='CnnPolicy')
model = make_ppo_agent(env, config=config, tensorboard_log='../logs/pixel_ppo')

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 256, but because the `RolloutBuffer` is of size `n_steps * n_envs = 128`, after every 0 untruncated mini-batches, there will be a truncated mini-batch of size 128
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=128 and n_envs=1)
  warnings.warn(


In [5]:
# Train
callback = CheckpointAndLogCallback(
    save_path='../models/pixel_ppo',
    save_freq=100_000,
)
model.learn(total_timesteps=config.total_timesteps, callback=callback)
model.save('../models/pixel_ppo/final_model')

Logging to ../logs/pixel_ppo/PPO_3


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(


----------------------------
| time/              |     |
|    fps             | 84  |
|    iterations      | 1   |
|    time_elapsed    | 1   |
|    total_timesteps | 128 |
----------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 229           |
|    ep_rew_mean          | 569           |
| time/                   |               |
|    fps                  | 88            |
|    iterations           | 2             |
|    time_elapsed         | 2             |
|    total_timesteps      | 256           |
| train/                  |               |
|    approx_kl            | 2.4633482e-07 |
|    clip_fraction        | 0             |
|    clip_range           | 0.1           |
|    entropy_loss         | -1.95         |
|    explained_variance   | 0             |
|    learning_rate        | 0.00025       |
|    loss                 | 2.64e+03      |
|    n_updates            | 4             |
|    

KeyboardInterrupt: 

In [ ]:
# Quick test
from src.utils.evaluation import evaluate_agent
results = evaluate_agent(model, env, n_episodes=10)
print(f"Mean reward: {results['mean_reward']:.1f} ± {results['std_reward']:.1f}")
print(f"Flag rate: {results['flag_rate']:.2%}")